# Toronto Airbnb Pricing Analytics\n**End-to-End Portfolio Project — Lambert Tan**\n\nThis notebook rebuilds the analysis from the November 2025 Toronto Inside Airbnb snapshot.

## 1. Business question\n**What drives Airbnb nightly prices in Toronto, and how can hosts use those drivers to set more defensible rates?**\n\nBecause the data are observational and cross-sectional, coefficients are interpreted as associations rather than causal effects.

In [ ]:
from pathlib import Path\nimport sys, numpy as np, pandas as pd, matplotlib.pyplot as plt\nROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()\nsys.path.insert(0, str(ROOT))\nfrom src.data_cleaning import clean_listings\nfrom src.feature_engineering import engineer_features\nfrom src.modeling import fit_final_model, coefficient_table, estimate_nightly_price\n

## 2. Load and audit the raw data\nThe source snapshot contains 21,468 listings and 79 fields.

In [ ]:
raw = pd.read_csv(ROOT / 'data/raw/toronto_listings_detail_nov.csv')\nprint(f'Raw shape: {raw.shape}')\nraw.head()\n

## 3. Data cleaning\nThe cleaning pipeline converts price to numeric, removes out-of-scope observations, parses bathrooms, removes inactive listings and keeps comparable room types.

In [ ]:
clean = clean_listings(raw)\nprint(f'Clean shape: {clean.shape}')\nclean.head()\n

## 4. Exploratory analysis\nNightly price is right-skewed, which motivates modelling log(price).

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))\naxes[0].hist(clean['price'], bins=50)\naxes[0].set(title='Nightly price', xlabel='CAD')\naxes[1].hist(np.log(clean['price']), bins=50)\naxes[1].set(title='Log nightly price', xlabel='log(CAD)')\nplt.tight_layout(); plt.show()\n

## 5. Feature engineering\nThe project creates amenity count, host experience, distance to Union Station and interpretable binary indicators.

In [ ]:
data = engineer_features(clean)\nprint(f'Engineered shape: {data.shape}')\ndata.describe().T.round(2)\n

## 6. Model development and validation\nThe final specification uses a 70/30 train/test split. HC3 robust standard errors are used after the Breusch–Pagan test identifies heteroskedasticity.

In [ ]:
model, robust_model, metrics, bp = fit_final_model(data)\nresults = coefficient_table(model, robust_model)\nprint(metrics)\nprint(bp)\nresults.round(4)\n

## 7. Business interpretation\nProperty setup is the foundation of price, downtown distance is a secondary adjustment, and operational features provide smaller uplifts.

In [ ]:
display_vars=['is_entire_home','is_shared_bath','distance_to_downtown_km','instant_bookable','amenity_count']\nresults.loc[display_vars,['coefficient','robust_pvalue','percent_impact']].round(3)\n

## 8. Pricing scenario\nThe fitted model can be used as a simple scenario tool.

In [ ]:
scenario=dict(accommodates=4,bedrooms=2,bathrooms=1,is_shared_bath=0,is_entire_home=1,distance_to_downtown_km=5,amenity_count=35,instant_bookable=1)\nestimate=estimate_nightly_price(model,**scenario)\nprint(f'Illustrative model estimate: {estimate:,.0f} CAD per night')\n

## 9. Recommendations\n1. **Benchmark property fundamentals first.**\n2. **Use location as an adjustment.**\n3. **Use operational features for incremental gains.**\n4. **Treat Superhost as a trust signal, not a direct price premium.**

## 10. Limitations\nThis is a single-market, single-snapshot observational analysis. It does not establish causality and does not directly model occupancy, booking conversion, seasonality, events, transit travel time or live competitor prices.